# Lab 1: Load a CSV into PostgreSQL

**Week 1 · Data Engineering Course**

---

## Objectives

By the end of this lab you will:

1. Understand the CSV schema before creating a table
2. Create a table in PostgreSQL with the correct column types
3. Load `employees.csv` into that table using pgAdmin
4. Verify the data loaded correctly
5. Write basic SQL queries against the live data

**Time estimate:** 30–45 minutes

**Tool:** All SQL runs in **pgAdmin Query Tool** (Tools → Query Tool)

---

## Setup

Before starting:

1. Open **pgAdmin** from the Start menu
2. Enter your master password
3. Connect to **PostgreSQL 16** (enter your `postgres` password if prompted)
4. In the left panel, click on **`de_course`** to select it
5. Click **Tools → Query Tool** — this is where all SQL in this lab runs

> Make sure `de_course` is shown in the database dropdown at the top of the Query Tool.

---

## Step 1: Inspect the CSV

Before creating a table, understand what's in the file.

Open **`week1/data/employees.csv`** in Notepad or any text editor. The first few lines look like this:

```
employee_id,first_name,last_name,department,salary,hire_date,city,is_active
1,Alice,Johnson,Engineering,95000,2021-03-15,New York,true
2,Bob,Smith,Marketing,72000,2020-07-01,Chicago,true
3,Carol,Williams,Engineering,110000,2019-11-20,San Francisco,true
4,David,Brown,Sales,68000,2022-01-10,Chicago,true
```

The file has **8 columns** and **20 data rows** (plus a header row).

---

### Quick data type reference

A **data type** tells PostgreSQL what kind of values a column stores. Every column must have one.

| Type | What it stores | Examples |
|------|---------------|----------|
| `INTEGER` | Whole numbers | `1`, `95000`, `-7` |
| `VARCHAR(n)` | Text up to n characters | `'Alice'`, `'Engineering'` |
| `DATE` | Calendar dates | `2021-03-15` |
| `BOOLEAN` | True or false | `true`, `false` |

Full reference of all PostgreSQL types is in Lesson 4 Section 4.7.

---

### Column → Data type mapping

| Column | Sample values | PostgreSQL type | Reason |
|--------|--------------|-----------------|--------|
| `employee_id` | 1, 2, 3 | `INTEGER` | Whole number, unique per employee |
| `first_name` | Alice, Bob | `VARCHAR(50)` | Short text, max 50 characters |
| `last_name` | Johnson, Smith | `VARCHAR(50)` | Short text |
| `department` | Engineering | `VARCHAR(50)` | Short text |
| `salary` | 95000, 72000 | `INTEGER` | Whole dollar amount |
| `hire_date` | 2021-03-15 | `DATE` | Calendar date (YYYY-MM-DD format) |
| `city` | New York | `VARCHAR(100)` | Slightly longer — city names can be long |
| `is_active` | true, false | `BOOLEAN` | Two states: active or inactive |

---

## Step 2: Create the Table

### What is a PRIMARY KEY?

A **primary key** is a column that uniquely identifies every row in a table:
- No two rows can have the same primary key value
- It can never be NULL (empty)
- Every well-designed table has one

In our employees table, `employee_id` is the primary key because every employee has a unique ID.

---

In the **pgAdmin Query Tool**, paste the following and press **F5**:

```sql
CREATE TABLE employees (
    employee_id  INTEGER      PRIMARY KEY,
    first_name   VARCHAR(50)  NOT NULL,
    last_name    VARCHAR(50)  NOT NULL,
    department   VARCHAR(50),
    salary       INTEGER,
    hire_date    DATE,
    city         VARCHAR(100),
    is_active    BOOLEAN
);
```

You should see `CREATE TABLE` in the **Messages** tab.

- `PRIMARY KEY` — uniquely identifies each row; no duplicates, no NULLs
- `NOT NULL` — this column must always have a value
- Columns without `NOT NULL` can be left empty (NULL) in the data

---

### Verify the table was created

```sql
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'employees'
ORDER BY ordinal_position;
```

This should return 8 rows — one for each column.

You can also see it visually in the left panel:
**de_course → Schemas → public → Tables → employees**
(You may need to right-click Tables → Refresh to see it appear.)

---

## Step 3: Load the CSV

We'll use **pgAdmin's Import/Export feature** — the easiest way to load a CSV on Windows.

### Instructions

1. In the left panel, expand **de_course → Schemas → public → Tables**
2. Right-click **`employees`**
3. Choose **Import/Export Data...**

4. Fill in the dialog:

   | Field | Value |
   |-------|-------|
   | Import/Export | **Import** |
   | Filename | Browse to `week1\data\employees.csv` on your computer |
   | Format | **csv** |
   | Encoding | UTF8 |

5. Click the **Options** tab:

   | Field | Value |
   |-------|-------|
   | Header | **Yes** (the file has a header row) |
   | Delimiter | `,` |

6. Click **OK**

You should see a success message: **`Successfully completed.`**

---

### Alternative: COPY command in Query Tool

If you prefer SQL, paste this in the Query Tool and update the file path to match where your file is saved:

```sql
COPY employees (employee_id, first_name, last_name, department, salary, hire_date, city, is_active)
FROM 'C:\Users\YourName\lessons\week1\data\employees.csv'
DELIMITER ','
CSV HEADER;
```

> **Windows path tip:** Use either backslashes `\` or forward slashes `/` — PostgreSQL accepts both.  
> Replace `YourName` with your actual Windows username.

Expected output in the Messages tab:
```
COPY 20
```

---

## Step 4: Verify the Load

Run each query in pgAdmin and check the expected results.

**Count the rows — expect 20:**

```sql
SELECT COUNT(*) AS total_rows
FROM employees;
```

---

**Preview the first 5 rows:**

```sql
SELECT *
FROM employees
LIMIT 5;
```

---

**Check for NULLs in key columns — both should return 0:**

```sql
SELECT COUNT(*) AS null_first_names
FROM employees
WHERE first_name IS NULL;
```

```sql
SELECT COUNT(*) AS null_employee_ids
FROM employees
WHERE employee_id IS NULL;
```

---

**Browse data without SQL:**

Right-click the `employees` table in the left panel → **View/Edit Data → First 100 Rows**  
pgAdmin shows the data as a spreadsheet automatically.

---

## Step 5: Exploration Queries

Run each query in pgAdmin and observe the output.

---

### 5a. Count employees by department

```sql
SELECT department, COUNT(*) AS headcount
FROM employees
GROUP BY department
ORDER BY headcount DESC;
```

> `GROUP BY` and `COUNT()` are covered in Week 2. Run it now and observe the pattern — each department name appears once with a count of how many employees are in it.

---

### 5b. Highest-paid employee per department

```sql
SELECT department, MAX(salary) AS max_salary
FROM employees
GROUP BY department
ORDER BY max_salary DESC;
```

> `MAX()` returns the largest value in a group. Again, covered in Week 2 — observe now.

---

### 5c. Active Engineering employees sorted by salary

```sql
SELECT first_name, last_name, salary, hire_date
FROM employees
WHERE department = 'Engineering'
  AND is_active = TRUE
ORDER BY salary DESC;
```

---

### 5d. Employees hired in 2022

```sql
SELECT first_name, last_name, hire_date, department
FROM employees
WHERE hire_date >= '2022-01-01'
  AND hire_date < '2023-01-01'
ORDER BY hire_date;
```

---

### 5e. Unique cities

```sql
SELECT DISTINCT city
FROM employees
ORDER BY city;
```

---

## Step 6: Your Own Queries

Write SQL in pgAdmin to answer each question, test it, then paste your working query into the cell below it.

> The cells below are **text cells** — they will not run. Paste your SQL as plain text.

**Q1: Which employees in Sales earn more than $70,000?**

**Q2: Show the 5 most recently hired employees.**

**Q3: How many distinct cities do employees work in?**

**Q4: List all inactive employees (`is_active = false`) sorted by last name.**

**Q5: Which employees work in New York and earn more than $80,000?**

---

## Troubleshooting

**Import/Export dialog shows an error about file not found**  
Make sure you browsed to the correct file. The path should end with `employees.csv`.  
Try moving the file to your Desktop to simplify the path.

---

**`ERROR: duplicate key value violates unique constraint "employees_pkey"`**  
The table already has data. Clear it first and reload:

```sql
TRUNCATE TABLE employees;
```

Then repeat the Import/Export step.

---

**`ERROR: invalid input syntax for type date`**  
The CSV date format doesn't match. Check that `hire_date` values look exactly like `2021-03-15` (YYYY-MM-DD).

---

**`ERROR: invalid input syntax for type boolean`**  
Run this in Query Tool to fix the column after loading:
```sql
ALTER TABLE employees ALTER COLUMN is_active TYPE BOOLEAN
USING is_active::boolean;
```

---

**Table not visible in the left panel**  
Right-click **Tables** → **Refresh**.

---

## Solutions — Step 6

Check your answers after attempting all five questions.

---

**Q1 — Sales employees earning > $70,000**

```sql
SELECT first_name, last_name, salary
FROM employees
WHERE department = 'Sales'
  AND salary > 70000
ORDER BY salary DESC;
```

---

**Q2 — 5 most recently hired**

```sql
SELECT first_name, last_name, hire_date, department
FROM employees
ORDER BY hire_date DESC
LIMIT 5;
```

---

**Q3 — Count of distinct cities**

```sql
SELECT COUNT(DISTINCT city) AS num_cities
FROM employees;
```

---

**Q4 — Inactive employees sorted by last name**

```sql
SELECT first_name, last_name, department, hire_date
FROM employees
WHERE is_active = FALSE
ORDER BY last_name ASC;
```

---

**Q5 — New York employees earning > $80,000**

```sql
SELECT first_name, last_name, salary, department
FROM employees
WHERE city = 'New York'
  AND salary > 80000
ORDER BY salary DESC;
```